# 23_Packet_P006_Panel_Candidate_Profiling
## Materia Arche Agent OS v3.0

**Work Packet ID:** P-006
**Decision this packet informs:** Are the P-005 lab panel candidates compositionally diverse enough for a meaningful Phase 2 test?
**Fastest falsifier:** Profile all 3 candidates — if they share the same composition and likely come from the same paper, the panel tests one point in composition space, not three.
**Success threshold:** Panel covers at least 2 distinct composition families
**Failure / kill threshold:** All 3 candidates are effectively the same composition
**Reuse requirement if it fails:** Recommend a diversified panel from the P-005 qualified candidate pool (31 devices)
**Dataset ID/version:** perovskite_stability_clean.csv (1,543 devices)
**Benchmark ID:** P-005 robust lab panel (devices 850, 1374, 848)
**Owner:** ML Scientist
**Reviewer:** Evidence Guardian
**Release ceiling:** Internal

In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.model_selection import train_test_split
from scipy.stats import kendalltau
from collections import Counter
import yaml
import warnings
warnings.filterwarnings('ignore')

# Load lock files
with open('contracts/schema.lock.yaml') as f:
    schema = yaml.safe_load(f)
with open('contracts/model.lock.yaml') as f:
    model_lock = yaml.safe_load(f)

print("Libraries loaded — Packet P-006")
print(f"Schema: {schema['dataset_id']} v{schema['version']}")
print(f"Model: {model_lock['model_id']}")

Libraries loaded — Packet P-006
Schema: perovskite_stability_clean vv1
Model: ET_baseline_v1


## 1. Load data and profile P-005 panel candidates

In [2]:
df = pd.read_csv('perovskite_stability_clean.csv')
print(f"Dataset: {len(df)} devices")

# P-005 panel
panel_indices = [850, 1374, 848]

# Composition columns
comp_cols = ['Perovskite_band_gap', 'Pb', 'Sn', 'I', 'Br', 'Cl', 'MA', 'FA', 'Cs']
process_cols = ['first_Prvskt_annealing_temperature', 'first_Prvskt_thermal_annealing_time',
                'Perovskite_thickness', 'Cell_area_measured']
device_cols = ['JV_default_Voc', 'JV_default_Jsc', 'JV_default_FF']

print("\n" + "=" * 80)
print("P-005 PANEL CANDIDATE PROFILES")
print("=" * 80)

for idx in panel_indices:
    row = df.iloc[idx]
    print(f"\nDevice {idx} — Actual T80: {row['Stability_PCE_T80']:.0f} hours")
    print(f"  Composition:")
    print(f"    A-site: MA={row['MA']:.2f}  FA={row['FA']:.2f}  Cs={row['Cs']:.2f}")
    print(f"    B-site: Pb={row['Pb']:.2f}  Sn={row['Sn']:.2f}")
    print(f"    X-site: I={row['I']:.2f}  Br={row['Br']:.2f}  Cl={row['Cl']:.2f}")
    print(f"    Bandgap: {row['Perovskite_band_gap']}")
    print(f"  Processing:")
    print(f"    Anneal temp: {row['first_Prvskt_annealing_temperature']}")
    print(f"    Anneal time: {row['first_Prvskt_thermal_annealing_time']}")
    print(f"    Thickness: {row['Perovskite_thickness']}")
    print(f"    Cell area: {row['Cell_area_measured']}")
    print(f"  Device performance:")
    print(f"    Voc: {row['JV_default_Voc']}")
    print(f"    Jsc: {row['JV_default_Jsc']}")
    print(f"    FF: {row['JV_default_FF']}")

Dataset: 1543 devices

P-005 PANEL CANDIDATE PROFILES

Device 850 — Actual T80: 3400 hours
  Composition:
    A-site: MA=3.00  FA=0.00  Cs=0.00
    B-site: Pb=4.00  Sn=0.00
    X-site: I=13.00  Br=0.00  Cl=0.00
    Bandgap: nan
  Processing:
    Anneal temp: 100.0
    Anneal time: 4.0
    Thickness: 350.0
    Cell area: 0.1
  Device performance:
    Voc: 1.01
    Jsc: 19.53
    FF: 0.764

Device 1374 — Actual T80: 1400 hours
  Composition:
    A-site: MA=1.00  FA=0.00  Cs=0.00
    B-site: Pb=1.00  Sn=0.00
    X-site: I=3.00  Br=0.00  Cl=0.00
    Bandgap: nan
  Processing:
    Anneal temp: 100.0
    Anneal time: 5.0
    Thickness: 300.0
    Cell area: 0.046
  Device performance:
    Voc: 0.95
    Jsc: 14.42
    FF: 0.66

Device 848 — Actual T80: 2200 hours
  Composition:
    A-site: MA=3.00  FA=0.00  Cs=0.00
    B-site: Pb=4.00  Sn=0.00
    X-site: I=13.00  Br=0.00  Cl=0.00
    Bandgap: nan
  Processing:
    Anneal temp: 100.0
    Anneal time: 4.0
    Thickness: 350.0
    Cell area: 0.1

## 2. Check compositional diversity

In [3]:
# Extract panel compositions
panel_df = df.iloc[panel_indices][comp_cols + process_cols + device_cols + ['Stability_PCE_T80']].copy()
panel_df.index = [f'Device_{i}' for i in panel_indices]

print("=" * 80)
print("COMPOSITIONAL DIVERSITY CHECK")
print("=" * 80)

# Check if compositions are identical
comp_only = df.iloc[panel_indices][comp_cols].fillna(0)
n_unique_compositions = len(comp_only.drop_duplicates())
print(f"\nUnique compositions in panel: {n_unique_compositions}/3")

# Classify each by A-site cation family
families = []
for idx in panel_indices:
    row = df.iloc[idx]
    ma, fa, cs = row['MA'], row['FA'], row['Cs']
    # Fill NaN with 0 for classification
    ma = ma if not np.isnan(ma) else 0
    fa = fa if not np.isnan(fa) else 0
    cs = cs if not np.isnan(cs) else 0
    
    if ma > 0 and fa == 0 and cs == 0:
        family = 'MA-only'
    elif fa > 0 and ma == 0 and cs == 0:
        family = 'FA-only'
    elif fa > 0 and ma > 0 and cs == 0:
        family = 'MA-FA mixed'
    elif fa > 0 and cs > 0:
        family = 'FA-Cs (triple)'
    elif ma > 0 and cs > 0:
        family = 'MA-Cs mixed'
    else:
        family = 'Other'
    families.append(family)
    print(f"  Device {idx}: {family} (MA={ma:.2f}, FA={fa:.2f}, Cs={cs:.2f})")

n_families = len(set(families))
print(f"\nDistinct composition families: {n_families}")
print(f"Success threshold: ≥2 families")

if n_families >= 2:
    print("PASS: Panel covers multiple composition families")
else:
    print(f"FAIL: All candidates are {families[0]} — panel needs diversification")

COMPOSITIONAL DIVERSITY CHECK

Unique compositions in panel: 2/3
  Device 850: MA-only (MA=3.00, FA=0.00, Cs=0.00)
  Device 1374: MA-only (MA=1.00, FA=0.00, Cs=0.00)
  Device 848: MA-only (MA=3.00, FA=0.00, Cs=0.00)

Distinct composition families: 1
Success threshold: ≥2 families
FAIL: All candidates are MA-only — panel needs diversification


## 3. Find diverse alternatives from P-005 qualified pool

If the panel lacks diversity, identify the best candidates from different composition families in the P-005 qualified pool (31 devices with ≥80% top-20 rate and T80 ≥500h).

In [4]:
# Reproduce P-005 test-set-only ranking to get the full qualified pool
X = df[schema['columns']].fillna(0)
y = np.log1p(df['Stability_PCE_T80'])

et_params = model_lock['params'].copy()
n_splits = 20
seeds = list(range(1, n_splits + 1))

device_test_appearances = Counter()
device_top20_counts = Counter()

for seed in seeds:
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=seed)
    et = ExtraTreesRegressor(random_state=42, **et_params)
    et.fit(X_tr, y_tr)
    pred_test = et.predict(X_te)
    test_ranks = pd.Series(pred_test, index=X_te.index).rank(ascending=False).astype(int)
    
    for idx in X_te.index:
        device_test_appearances[idx] += 1
        if test_ranks[idx] <= 20:
            device_top20_counts[idx] += 1

# Build qualified pool (same criteria as P-005)
pool_rows = []
for idx in range(len(df)):
    n_app = device_test_appearances[idx]
    if n_app < 3:
        continue
    n_top20 = device_top20_counts[idx]
    rate = n_top20 / n_app
    t80 = np.expm1(y.iloc[idx])
    if rate >= 0.80 and t80 >= 500:
        row = df.iloc[idx]
        ma = row['MA'] if not np.isnan(row['MA']) else 0
        fa = row['FA'] if not np.isnan(row['FA']) else 0
        cs = row['Cs'] if not np.isnan(row['Cs']) else 0
        
        if ma > 0 and fa == 0 and cs == 0:
            family = 'MA-only'
        elif fa > 0 and ma == 0 and cs == 0:
            family = 'FA-only'
        elif fa > 0 and ma > 0 and cs == 0:
            family = 'MA-FA mixed'
        elif fa > 0 and cs > 0:
            family = 'FA-Cs (triple)'
        elif ma > 0 and cs > 0:
            family = 'MA-Cs mixed'
        else:
            family = 'Other'
        
        pool_rows.append({
            'device_idx': idx,
            'actual_T80': t80,
            'top20_rate': rate,
            'top20_count': n_top20,
            'appearances': n_app,
            'family': family,
            'MA': ma, 'FA': fa, 'Cs': cs,
            'Pb': row['Pb'] if not np.isnan(row['Pb']) else 0,
            'Sn': row['Sn'] if not np.isnan(row['Sn']) else 0,
        })

pool = pd.DataFrame(pool_rows).set_index('device_idx')
pool = pool.sort_values(['top20_rate', 'actual_T80'], ascending=[False, False])

print(f"Qualified pool: {len(pool)} devices")
print(f"\nComposition families in pool:")
family_counts = pool['family'].value_counts()
for fam, count in family_counts.items():
    print(f"  {fam}: {count} devices")

Qualified pool: 31 devices

Composition families in pool:
  MA-only: 13 devices
  MA-FA mixed: 9 devices
  FA-Cs (triple): 7 devices
  Other: 2 devices


In [5]:
# Show best candidate from each family
print("=" * 80)
print("BEST CANDIDATE PER COMPOSITION FAMILY (from qualified pool)")
print("=" * 80)

family_best = {}
for fam in pool['family'].unique():
    fam_pool = pool[pool['family'] == fam].sort_values(['top20_rate', 'actual_T80'], ascending=[False, False])
    best = fam_pool.iloc[0]
    family_best[fam] = best
    print(f"\n  {fam}:")
    print(f"    Device {best.name}: T80={best['actual_T80']:.0f}h, top-20 rate={best['top20_rate']:.0%}")
    print(f"    A-site: MA={best['MA']:.2f} FA={best['FA']:.2f} Cs={best['Cs']:.2f}")
    print(f"    B-site: Pb={best['Pb']:.2f} Sn={best['Sn']:.2f}")

BEST CANDIDATE PER COMPOSITION FAMILY (from qualified pool)

  MA-FA mixed:
    Device 1064: T80=5400h, top-20 rate=100%
    A-site: MA=0.15 FA=0.85 Cs=0.00
    B-site: Pb=1.00 Sn=0.00

  MA-only:
    Device 1373: T80=5000h, top-20 rate=100%
    A-site: MA=1.00 FA=0.00 Cs=0.00
    B-site: Pb=1.00 Sn=0.00

  FA-Cs (triple):
    Device 119: T80=3423h, top-20 rate=100%
    A-site: MA=0.00 FA=0.83 Cs=0.17
    B-site: Pb=1.00 Sn=0.00

  Other:
    Device 228: T80=1032h, top-20 rate=100%
    A-site: MA=0.00 FA=0.00 Cs=1.00
    B-site: Pb=1.00 Sn=0.00


## 4. Recommend final panel

In [6]:
print("=" * 80)
print("PANEL RECOMMENDATION")
print("=" * 80)

if n_families >= 2:
    print("\nP-005 panel already has sufficient diversity.")
    print("Recommended panel (unchanged):")
    for i, idx in enumerate(panel_indices, 1):
        t80 = df.iloc[idx]['Stability_PCE_T80']
        print(f"  #{i} Device {idx}: T80={t80:.0f}h ({families[i-1]})")
    recommendation = "keep"
else:
    print(f"\nP-005 panel lacks diversity — all {families[0]}.")
    print("Recommended diversified panel:")
    
    # Keep the strongest P-005 candidate
    print(f"  #1 Device {panel_indices[0]}: T80={df.iloc[panel_indices[0]]['Stability_PCE_T80']:.0f}h ({families[0]}) [P-005 top pick]")
    
    # Add best from other families
    diversified = [panel_indices[0]]
    diversified_families = [families[0]]
    slot = 2
    
    for fam, best in sorted(family_best.items(), key=lambda x: (-x[1]['top20_rate'], -x[1]['actual_T80'])):
        if fam not in diversified_families and slot <= 3:
            diversified.append(best.name)
            diversified_families.append(fam)
            print(f"  #{slot} Device {best.name}: T80={best['actual_T80']:.0f}h ({fam}) [diversity pick]")
            slot += 1
    
    # If still need more, fill from remaining P-005 panel
    for idx in panel_indices[1:]:
        if idx not in diversified and slot <= 3:
            t80 = df.iloc[idx]['Stability_PCE_T80']
            diversified.append(idx)
            print(f"  #{slot} Device {idx}: T80={t80:.0f}h ({families[panel_indices.index(idx)]}) [P-005 backup]")
            slot += 1
    
    recommendation = "diversify"

print(f"\nRecommendation: {recommendation}")

PANEL RECOMMENDATION

P-005 panel lacks diversity — all MA-only.
Recommended diversified panel:
  #1 Device 850: T80=3400h (MA-only) [P-005 top pick]
  #2 Device 1064: T80=5400h (MA-FA mixed) [diversity pick]
  #3 Device 119: T80=3423h (FA-Cs (triple)) [diversity pick]

Recommendation: diversify


## 5. Honest status footer

In [7]:
if n_families >= 2:
    status = "Confirmed"
    decision = "Advance"
else:
    # Check if diversified panel is possible
    n_pool_families = len(pool['family'].unique())
    if n_pool_families >= 2:
        status = "Promising"
        decision = "Iterate"
    else:
        status = "Negative"
        decision = "Stop"

print("=" * 65)
print("HONEST STATUS — MATERIA ARCHE v3.0")
print("=" * 65)
print(f"Packet ID: P-006")
print(f"Status: {status}")
print(f"Evidence level: E3 (decision-grade shortlist)")
print(f"Decision outcome: {decision}")
print(f"Release ceiling: Internal")
print(f"Domain: perovskite")
print(f"Dataset version: perovskite_stability_clean.csv (1,543 devices)")
print(f"Benchmark: P-005 robust lab panel")
print(f"This run: {n_unique_compositions}/3 unique compositions, {n_families} family/families")

if n_families >= 2:
    print(f"What worked: Panel already covers multiple composition families")
    print(f"What failed or remains uncertain: Prediction intervals remain wide (P-003)")
    print(f"What changes next: Panel is ready for Phase 2 outreach")
else:
    print(f"What worked: Identified composition gap in P-005 panel")
    print(f"What failed or remains uncertain: P-005 top-3 are all {families[0]}")
    if n_pool_families >= 2:
        print(f"What changes next: Adopt diversified panel from qualified pool ({n_pool_families} families available)")
    else:
        print(f"What changes next: Only {families[0]} devices pass robustness — accept or collect more data")

print(f"Reusable asset created: Composition profiling of qualified pool")
print(f"Safeguard added: Composition diversity check before any Phase 2 outreach")
print(f"")
print(f"Reviewer sign-off: Evidence Guardian __________")

HONEST STATUS — MATERIA ARCHE v3.0
Packet ID: P-006
Status: Promising
Evidence level: E3 (decision-grade shortlist)
Decision outcome: Iterate
Release ceiling: Internal
Domain: perovskite
Dataset version: perovskite_stability_clean.csv (1,543 devices)
Benchmark: P-005 robust lab panel
This run: 2/3 unique compositions, 1 family/families
What worked: Identified composition gap in P-005 panel
What failed or remains uncertain: P-005 top-3 are all MA-only
What changes next: Adopt diversified panel from qualified pool (4 families available)
Reusable asset created: Composition profiling of qualified pool
Safeguard added: Composition diversity check before any Phase 2 outreach

Reviewer sign-off: Evidence Guardian __________
